# Prompt 09: Pilot run (Version A, T21)

Vaibhav Sharma, MSE2026010

What this notebook does: builds the frozen topic list, generates pilot answers from both models (6 pilot topics x 2 models = 12 answers), parses them into citations, runs B2 database matching **without a threshold**, and exports a blind sheet for hand labelling.

- Runtime: **T4 GPU** (Runtime > Change runtime type > T4 GPU).
- Secrets needed (key icon on the left): `OPENALEX_API_KEY`, `CONTACT_EMAIL`. `HF_TOKEN` is optional (both models are not gated).
- Expected time: about 30 to 50 minutes (two model downloads of about 15 GB each, then 6 answers per model).
- Every step writes its console output to `logs/`. Do not edit outputs. At the end, download `pilot_outputs.zip` and send it back.
- Pilot topics are used ONLY to fix the B2 threshold. They never enter the main analysis.

In [ ]:
# Cell 1: upload the repository zip (AITR_Lab02_MSE2026010_CiteVerify.zip) and unzip it
from google.colab import files
import zipfile, os
up = files.upload()
for n in up:
    if n.endswith('.zip'):
        zipfile.ZipFile(n).extractall('.')
os.chdir('AITR_Lab02_MSE2026010_CiteVerify')
os.makedirs('logs', exist_ok=True)
print(os.getcwd(), sorted(os.listdir('.')))

In [ ]:
# Cell 2: GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU', torch.cuda.is_available())

In [ ]:
# Cell 3: install what Colab lacks (same packages as the Prompt 08 environment test)
!pip install -q -r requirements.in
!pip check | tail -3

In [ ]:
# Cell 4: secrets from Colab Secrets (never typed into the notebook)
import os
from google.colab import userdata
for k in ('OPENALEX_API_KEY', 'CONTACT_EMAIL', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k) or ''
    except Exception:
        os.environ[k] = ''
    print(k, 'set' if os.environ[k] else 'NOT set')

In [ ]:
# Cell 5: unit tests (synthetic fixtures only; they say nothing about real models)
!python -m pytest -q 2>&1 | tee logs/p09_pytest.log | tail -5

In [ ]:
# Cell 6: frozen topic list (OpenAlex works count per topic, tercile bands, pilot topics flagged)
!python -m src.topics 2>&1 | tee logs/p09_topics.log | tail -15
import pandas as pd
t = pd.read_csv('data/topics_frozen.csv')
print(t.groupby(['band', 'is_pilot']).size())
print(t[t.is_pilot][['topic_id', 'topic', 'works_count', 'band']])

In [ ]:
# Cell 7: pilot generation, both models (resumable: if Colab disconnects, rerun cells 1 to 4 and this cell)
!python -m src.generate --pilot 2>&1 | tee logs/p09_generate_pilot.log | grep -v Warning | tail -20

In [ ]:
# Cell 8: look at one raw answer per model (for your own check of the output format)
from src.common import read_jsonl
g = read_jsonl('data/raw/generations_pilot.jsonl')
print(len(g), 'answers')
seen = set()
for r in g:
    if r['model'] not in seen:
        seen.add(r['model'])
        print('=' * 80, '\n', r['model'], r['topic'], r['n_new_tokens'], 'tokens', r['seconds'], 's\n')
        print(r['text'][:1500])

In [ ]:
# Cell 9: parse answers into citations
!python -m src.parse --mode pilot 2>&1 | tee logs/p09_parse_pilot.log
c = pd.DataFrame(read_jsonl('data/parsed/citations_pilot.jsonl'))
print(pd.crosstab(c['model'], c['parse_status']))

In [ ]:
# Cell 10: B2 evidence (CrossRef <= 3 requests/s + OpenAlex). No threshold yet, so no labels.
!python -m src.verify --mode pilot 2>&1 | tee logs/p09_verify_pilot.log | tail -5
e = pd.DataFrame(read_jsonl('data/labels/auto_pilot.jsonl'))
print(e[['best_sim', 'seconds', 'api_calls']].describe().round(3))
print('doi_resolves:', e['doi_resolves'].value_counts(dropna=False).to_dict())

In [ ]:
# Cell 11: blind labelling sheet (no model name, no B2 score)
!python scripts/export_labels.py --mode pilot 2>&1 | tee logs/p09_export.log

In [ ]:
# Cell 12: package everything and download
!zip -q -r pilot_outputs.zip logs data/topics_frozen.csv data/raw/generations_pilot.jsonl data/parsed/citations_pilot.jsonl data/labels/auto_pilot.jsonl data/labels/to_label_pilot.csv
!ls -la pilot_outputs.zip
files.download('pilot_outputs.zip')

## After this notebook (your part, by hand)

1. Open `data/labels/to_label_pilot.csv`. For each row, search the title yourself (Google Scholar, CrossRef, publisher site) and type one label in `gold_label`: `exists`, `metadata_error`, `fabricated` or `unresolvable`. Use `note` for anything unclear.
2. Save it as `gold_pilot.csv`. Keep a short note of how long labelling took and any hard cases.
3. Send `pilot_outputs.zip`, `gold_pilot.csv` and screenshots of cells 5 to 11. Prompt 10 then fixes the B2 threshold with `python -m src.analyse_a --pilot`.